# Analisis RFM y KMeans

Notebook de exploracion para el dataset sintetico de e-commerce. El flujo replica el analisis del script `src/rfm_kmeans_analysis.py`, pero separado por celdas para poder inspeccionar cada paso.

> Advertencia: los datos son sinteticos. Los resultados sirven para aprendizaje, prototipado y validacion tecnica, no para decisiones de negocio reales.

## 1. Setup

Importamos librerias, resolvemos rutas del proyecto y cargamos las funciones reutilizables del pipeline.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from rfm_kmeans_analysis import (
    add_kmeans_segments,
    choose_cluster_count,
    evaluate_kmeans_elbow,
    build_order_level_dataset,
    build_rfm,
    clean_data,
    read_data,
)

pd.options.display.float_format = "{:,.2f}".format

PROJECT_ROOT

WindowsPath('c:/Users/jptru/platzi/spikes/spike-vscode-ia-datos')

## 2. Diccionario de datos

Antes de transformar, revisamos columnas, relaciones y reglas relevantes documentadas en `data/data_dictionary.csv`.

In [2]:
data_dictionary = pd.read_csv(PROJECT_ROOT / "data" / "data_dictionary.csv")
data_dictionary

,table,column,description,example,rules
0,customers,customer_id,Identificador de cliente a nivel orden. Imita ...,col_customer_0000123,Llave primaria de customers. Debe ser unica. S...
1,customers,customer_unique_id,Identificador persistente de persona/comprador.,col_unique_000045,"Usar para RFM, frecuencia, recompra y segmenta..."
2,customers,customer_city,Ciudad del cliente en Colombia.,Medellin,Puede tener nulos controlados. Normalizar text...
3,customers,customer_department,Departamento del cliente.,Antioquia,Puede tener nulos controlados. Sirve para filt...
4,customers,customer_postal_code_prefix,Prefijo postal sintetico de tres digitos.,050,Puede tener nulos controlados. Tratar como tex...
5,orders,order_id,Identificador unico de orden.,col_order_0000123,Llave primaria de orders. Se une con order_ite...
6,orders,customer_id,Cliente asociado a la orden.,col_customer_0000123,Llave foranea hacia customers.customer_id. Cad...
7,orders,order_status,Estado operativo de la orden.,delivered,"Valores esperados: delivered, canceled, return..."
8,orders,order_purchase_timestamp,Fecha y hora de compra.,2024-08-15 14:32:10,Convertir con pd.to_datetime. Base para series...
9,orders,order_approved_at,Fecha y hora de aprobacion del pago/orden.,2024-08-15 15:20:10,Puede ser nula si la orden queda pendiente. No...


## 3. Carga de datos

Cargamos los cuatro CSV principales: clientes, ordenes, items y pagos.

In [3]:
tables_raw = read_data()

for table_name, df in tables_raw.items():
    print(f"{table_name}: {df.shape[0]:,} filas x {df.shape[1]:,} columnas")

customers: 5,988 filas x 5 columnas
orders: 5,988 filas x 9 columnas
order_items: 9,238 filas x 9 columnas
order_payments: 5,980 filas x 6 columnas


In [4]:
tables_raw["orders"].head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,currency
0,col_order_0000001,col_customer_0000001,delivered,2024-09-17 03:38:41,2024-09-18 22:36:41,2024-09-21 21:36:41,2024-09-25 12:36:41,2024-09-27 03:38:41,COP
1,col_order_0000002,col_customer_0000002,canceled,2024-09-21 02:15:21,2024-09-22 15:31:21,NaN,NaN,2024-10-04 02:15:21,COP
2,col_order_0000003,col_customer_0000003,delivered,2025-04-20 02:21:59,2025-04-21 08:03:59,2025-04-24 13:03:59,2025-04-26 07:03:59,2025-04-25 02:21:59,COP
3,col_order_0000004,col_customer_0000004,payment_pending,2025-06-26 22:43:08,NaN,NaN,NaN,2025-07-02 22:43:08,COP
4,col_order_0000005,col_customer_0000005,delivered,2025-04-13 15:06:47,2025-04-14 02:50:47,2025-04-16 12:50:47,2025-04-29 01:50:47,2025-04-21 15:06:47,COP


## 4. Limpieza basica

Aplicamos las reglas principales del diccionario: fechas a `datetime`, textos normalizados, nulos controlados y pagos duplicados removidos.

In [5]:
tables = clean_data(tables_raw)

cleaning_summary = pd.DataFrame(
    {
        "tabla": list(tables.keys()),
        "filas": [len(df) for df in tables.values()],
        "columnas": [df.shape[1] for df in tables.values()],
        "nulos_totales": [df.isna().sum().sum() for df in tables.values()],
    }
)

cleaning_summary

,tabla,filas,columnas,nulos_totales
0,customers,5988,5,94
1,orders,5988,9,2353
2,order_items,9238,9,0
3,order_payments,5970,6,0


## 5. Dataset maestro a nivel orden

Agregamos items y pagos a grano de orden, y luego unimos los atributos del cliente. Esto evita duplicar ventas por la relacion 1:N entre ordenes e items/pagos.

In [6]:
order_level = build_order_level_dataset(tables)
order_level.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,currency,customer_unique_id,...,customer_postal_code_prefix,item_count,gross_merchandise_value,freight_value,unique_products,main_category,monetary,payment_count,max_installments,main_payment_type
0,col_order_0000001,col_customer_0000001,delivered,2024-09-17 03:38:41,2024-09-18 22:36:41,2024-09-21 21:36:41,2024-09-25 12:36:41,2024-09-27 03:38:41,COP,col_unique_000001,...,130,3.00,"940,100.00","84,800.00",3.00,celulares,"1,024,900.00",1.00,1.00,transferencia
1,col_order_0000002,col_customer_0000002,canceled,2024-09-21 02:15:21,2024-09-22 15:31:21,NaT,NaT,2024-10-04 02:15:21,COP,col_unique_000002,...,080,1.00,"113,600.00","19,900.00",1.00,moda,"133,500.00",1.00,1.00,tarjeta_debito
2,col_order_0000003,col_customer_0000003,delivered,2025-04-20 02:21:59,2025-04-21 08:03:59,2025-04-24 13:03:59,2025-04-26 07:03:59,2025-04-25 02:21:59,COP,col_unique_000003,...,660,2.00,"290,500.00","34,700.00",2.00,juguetes,"325,200.00",1.00,1.00,transferencia
3,col_order_0000004,col_customer_0000004,payment_pending,2025-06-26 22:43:08,NaT,NaT,NaT,2025-07-02 22:43:08,COP,col_unique_000004,...,500,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>
4,col_order_0000005,col_customer_0000005,delivered,2025-04-13 15:06:47,2025-04-14 02:50:47,2025-04-16 12:50:47,2025-04-29 01:50:47,2025-04-21 15:06:47,COP,col_unique_000005,...,050,2.00,"369,700.00","30,000.00",2.00,hogar,"399,700.00",1.00,1.00,transferencia


In [7]:
order_level[[
    "order_id",
    "customer_unique_id",
    "customer_department",
    "order_status",
    "order_purchase_timestamp",
    "item_count",
    "monetary",
]].sample(10, random_state=42)

,order_id,customer_unique_id,customer_department,order_status,order_purchase_timestamp,item_count,monetary
3574,col_order_0003575,col_unique_002497,bogota d.c.,delivered,2024-05-31 08:18:12,1.00,"118,400.00"
5792,col_order_0005793,col_unique_004066,atlantico,returned,2024-06-04 22:42:02,1.00,"82,200.00"
4162,col_order_0004163,col_unique_002919,quindio,canceled,2025-05-11 15:19:42,2.00,"350,800.00"
1609,col_order_0001610,col_unique_001130,bogota d.c.,returned,2025-05-24 07:59:31,1.00,"134,200.00"
3727,col_order_0003728,col_unique_002609,bogota d.c.,processing,2025-07-20 22:38:15,1.00,"92,200.00"
1042,col_order_0001043,col_unique_000716,bogota d.c.,delivered,2024-12-15 23:53:19,1.00,"141,000.00"
4427,col_order_0004428,col_unique_003110,caldas,delivered,2024-10-30 03:31:05,1.00,"566,400.00"
5217,col_order_0005218,col_unique_003656,valle del cauca,delivered,2025-03-03 23:58:54,1.00,"167,900.00"
5468,col_order_0005469,col_unique_003843,quindio,delivered,2025-04-03 04:31:11,1.00,"116,800.00"
1485,col_order_0001486,col_unique_001040,valle del cauca,returned,2025-04-25 11:40:56,3.00,"558,000.00"


## 6. KPIs de ordenes entregadas

Para ventas efectivas usamos solo ordenes con `order_status == delivered`, como recomienda el diccionario de datos.

In [8]:
delivered = order_level.loc[order_level["order_status"] == "delivered"].copy()
delivered["monetary"] = delivered["monetary"].fillna(0)

total_sales = delivered["monetary"].sum()
total_orders = delivered["order_id"].nunique()
total_customers = delivered["customer_unique_id"].nunique()
avg_ticket = total_sales / total_orders

pd.DataFrame(
    {
        "kpi": ["ventas_entregadas", "ordenes_entregadas", "clientes_unicos", "ticket_promedio"],
        "valor": [total_sales, total_orders, total_customers, avg_ticket],
    }
)

,kpi,valor
0,ventas_entregadas,"1,833,881,900.00"
1,ordenes_entregadas,"4,136.00"
2,clientes_unicos,"3,143.00"
3,ticket_promedio,"443,395.04"


## 7. Ventas por mes

Serie mensual de ventas entregadas usando el valor pagado (`payment_value`) agregado a nivel orden como `monetary`.

In [9]:
delivered["purchase_month"] = delivered["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp()

monthly_sales = (
    delivered.groupby("purchase_month", as_index=False)
    .agg(
        sales=("monetary", "sum"),
        orders=("order_id", "nunique"),
        customers=("customer_unique_id", "nunique"),
    )
    .sort_values("purchase_month")
)

monthly_sales.head()

,purchase_month,sales,orders,customers
0,2024-01-01,"61,026,000.00",146,146
1,2024-02-01,"94,273,100.00",172,171
2,2024-03-01,"70,782,700.00",168,168
3,2024-04-01,"79,047,100.00",169,169
4,2024-05-01,"76,427,100.00",174,174


In [10]:
fig_sales = px.line(
    monthly_sales,
    x="purchase_month",
    y="sales",
    markers=True,
    title="Ventas entregadas por mes",
    labels={"purchase_month": "Mes", "sales": "Ventas COP", "orders": "Ordenes"},
    hover_data={"orders": True, "customers": True, "sales": ":,.0f"},
)
fig_sales.update_layout(template="plotly_white", hovermode="x unified")
fig_sales.show()

## 8. Analisis por departamento

Vista agregada para encontrar los departamentos con mayor venta entregada.

In [11]:
department_sales = (
    delivered.groupby("customer_department", as_index=False)
    .agg(
        sales=("monetary", "sum"),
        orders=("order_id", "nunique"),
        customers=("customer_unique_id", "nunique"),
    )
    .sort_values("sales", ascending=False)
)

department_sales.head(10)

,customer_department,sales,orders,customers
2,bogota d.c.,"465,330,500.00",1035,784
0,antioquia,"294,342,600.00",606,450
20,valle del cauca,"159,014,000.00",372,295
1,atlantico,"141,578,300.00",293,234
17,santander,"92,792,900.00",231,174
3,bolivar,"73,449,700.00",195,151
9,cundinamarca,"72,514,200.00",178,142
16,risaralda,"62,052,800.00",126,98
5,caldas,"56,663,900.00",128,91
14,norte de santander,"56,526,000.00",124,90


In [12]:
fig_dept = px.bar(
    department_sales.head(15),
    x="sales",
    y="customer_department",
    orientation="h",
    title="Top departamentos por ventas entregadas",
    labels={"sales": "Ventas COP", "customer_department": "Departamento"},
    hover_data={"orders": True, "customers": True, "sales": ":,.0f"},
)
fig_dept.update_layout(template="plotly_white", yaxis={"categoryorder": "total ascending"})
fig_dept.show()

## 9. RFM

Calculamos recencia, frecuencia y valor monetario por `customer_unique_id`, que es el identificador correcto para medir recompra y segmentacion.

In [13]:
rfm = build_rfm(order_level)
rfm.head()

,customer_unique_id,recency,frequency,monetary,r_score,f_score,m_score,rfm_score
0,col_unique_000001,454,1,"1,024,900.00",2,1,4,214
1,col_unique_000003,239,1,"325,200.00",3,1,2,312
2,col_unique_000005,246,1,"399,700.00",3,1,3,313
3,col_unique_000006,273,2,"452,300.00",3,4,3,343
4,col_unique_000007,553,1,"661,700.00",1,1,3,113


In [14]:
rfm[["recency", "frequency", "monetary"]].describe()

,recency,frequency,monetary
count,"3,143.00","3,143.00","3,143.00"
mean,384.22,1.32,"583,481.36"
std,181.07,0.71,"605,277.65"
min,1.00,1.00,"59,700.00"
25%,238.00,1.00,"168,150.00"
50%,381.00,1.00,"380,500.00"
75%,534.50,1.00,"802,500.00"
max,714.00,7.00,"7,466,100.00"


## 10. Validacion del numero de clusters

Antes de entrenar KMeans validamos varios valores de `k` con el metodo del codo. La idea es encontrar el punto donde agregar mas clusters deja de reducir la inercia de forma material.

La seleccion automatica usa la mayor distancia entre cada punto de la curva y la linea que une el primer y ultimo punto evaluado. Esto no reemplaza criterio de negocio, pero da una base mas defendible que fijar `k` manualmente.

In [15]:
elbow_results = evaluate_kmeans_elbow(rfm, min_clusters=2, max_clusters=10)
selected_k = choose_cluster_count(elbow_results, fallback=4)

print(f"Numero de clusters recomendado por codo: {selected_k}")
elbow_results

Numero de clusters recomendado por codo: 5


,k,inertia,inertia_drop_pct,elbow_distance
0,2,"5,905.14",NaN,0.00
1,3,"4,140.38",29.89,0.20
2,4,"3,346.69",19.17,0.24
3,5,"2,744.04",18.01,0.25
4,6,"2,468.66",10.04,0.20
5,7,"2,209.73",10.49,0.16
6,8,"1,942.67",12.09,0.11
7,9,"1,735.43",10.67,0.06
8,10,"1,541.42",11.18,0.00


In [16]:
fig_elbow = px.line(
    elbow_results,
    x="k",
    y="inertia",
    markers=True,
    title="Metodo del codo para KMeans",
    labels={"k": "Numero de clusters", "inertia": "Inercia"},
    hover_data={"inertia_drop_pct": ":.2f", "elbow_distance": ":.4f"},
)

fig_elbow.add_vline(
    x=selected_k,
    line_dash="dash",
    line_color="red",
    annotation_text=f"k recomendado = {selected_k}",
)
fig_elbow.update_layout(template="plotly_white")
fig_elbow.show()

## 11. Segmentacion KMeans con k seleccionado

Con el `k` recomendado por el codo, volvemos a entrenar KMeans sobre las variables RFM escaladas.

In [17]:
segments = add_kmeans_segments(rfm, n_clusters=selected_k)
segments.head()

,customer_unique_id,recency,frequency,monetary,r_score,f_score,m_score,rfm_score,cluster,segment_name
0,col_unique_000001,454,1,"1,024,900.00",2,1,4,214,4,segmento_2
1,col_unique_000003,239,1,"325,200.00",3,1,2,312,0,segmento_4
2,col_unique_000005,246,1,"399,700.00",3,1,3,313,0,segmento_4
3,col_unique_000006,273,2,"452,300.00",3,4,3,343,3,segmento_3
4,col_unique_000007,553,1,"661,700.00",1,1,3,113,2,segmento_5


In [18]:
segment_summary = (
    segments.groupby(["cluster", "segment_name"], as_index=False)
    .agg(
        customers=("customer_unique_id", "count"),
        avg_recency=("recency", "mean"),
        avg_frequency=("frequency", "mean"),
        avg_monetary=("monetary", "mean"),
        total_monetary=("monetary", "sum"),
    )
    .sort_values("avg_monetary", ascending=False)
)

segment_summary

,cluster,segment_name,customers,avg_recency,avg_frequency,avg_monetary,total_monetary
1,1,segmento_1,117,130.32,3.84,"2,260,838.46","264,518,100.00"
4,4,segmento_2,365,448.65,1.25,"1,383,331.78","504,916,100.00"
3,3,segmento_3,421,216.32,2.24,"757,202.14","318,782,100.00"
0,0,segmento_4,1123,272.52,1.00,"335,562.51","376,836,700.00"
2,2,segmento_5,1117,565.33,1.04,"330,195.97","368,828,900.00"


## 12. Interpretacion de negocio

Traducimos los clusters tecnicos a segmentos accionables. Para marketing, lo importante no es el numero del cluster, sino que decision habilita: retener, desarrollar, reactivar o cuidar inversion comercial.

In [19]:
median_recency = segment_summary["avg_recency"].median()
high_recency = segment_summary["avg_recency"].quantile(0.75)
median_frequency = segment_summary["avg_frequency"].median()
median_monetary = segment_summary["avg_monetary"].median()
high_monetary = segment_summary["avg_monetary"].quantile(0.75)

def explain_segment(row):
    is_recent = row["avg_recency"] <= median_recency
    is_inactive = row["avg_recency"] >= high_recency
    is_frequent = row["avg_frequency"] >= median_frequency
    is_high_value = row["avg_monetary"] >= high_monetary
    is_mid_value = row["avg_monetary"] >= median_monetary

    if is_recent and is_frequent and is_high_value:
        return pd.Series({
            "business_label": "VIP recientes",
            "vp_marketing_read": "Clientes de alto valor, compra reciente y mayor recurrencia relativa. Son la base de retencion premium.",
            "recommended_action": "Proteger margen y lealtad: beneficios selectivos, acceso anticipado, cross-sell premium y programas de fidelizacion.",
        })

    if is_inactive and is_mid_value:
        return pd.Series({
            "business_label": "Alto valor en riesgo",
            "vp_marketing_read": "Compraron bien, pero llevan mas tiempo sin volver. Hay valor historico que puede perderse.",
            "recommended_action": "Campanas win-back con incentivo controlado, mensajes personalizados y recordatorios por categoria comprada.",
        })

    if is_recent and is_mid_value:
        return pd.Series({
            "business_label": "Activos con potencial",
            "vp_marketing_read": "Clientes recientes con buen gasto relativo, listos para aumentar frecuencia o ticket.",
            "recommended_action": "Impulsar segunda compra, bundles, recomendaciones y comunicaciones por afinidad de categoria.",
        })

    if is_inactive and not is_frequent:
        return pd.Series({
            "business_label": "Dormidos de bajo retorno",
            "vp_marketing_read": "Baja frecuencia y poca recencia. Conviene no sobreinvertir hasta validar respuesta.",
            "recommended_action": "Reactivacion de bajo costo, audiencias lookalike separadas y supresion gradual si no responden.",
        })

    return pd.Series({
        "business_label": "Base estable",
        "vp_marketing_read": "Clientes con comportamiento intermedio. Funcionan como base para nutricion y experimentos comerciales.",
        "recommended_action": "Automatizaciones always-on, pruebas A/B de oferta y educacion de categorias complementarias.",
    })

business_explanation = pd.concat(
    [segment_summary.reset_index(drop=True), segment_summary.apply(explain_segment, axis=1).reset_index(drop=True)],
    axis=1,
)

segments = segments.merge(
    business_explanation[["cluster", "business_label", "vp_marketing_read", "recommended_action"]],
    on="cluster",
    how="left",
)

business_explanation

,cluster,segment_name,customers,avg_recency,avg_frequency,avg_monetary,total_monetary,business_label,vp_marketing_read,recommended_action
0,1,segmento_1,117,130.32,3.84,"2,260,838.46","264,518,100.00",VIP recientes,"Clientes de alto valor, compra reciente y mayo...",Proteger margen y lealtad: beneficios selectiv...
1,4,segmento_2,365,448.65,1.25,"1,383,331.78","504,916,100.00",Alto valor en riesgo,"Compraron bien, pero llevan mas tiempo sin vol...","Campanas win-back con incentivo controlado, me..."
2,3,segmento_3,421,216.32,2.24,"757,202.14","318,782,100.00",Activos con potencial,"Clientes recientes con buen gasto relativo, li...","Impulsar segunda compra, bundles, recomendacio..."
3,0,segmento_4,1123,272.52,1.00,"335,562.51","376,836,700.00",Base estable,Clientes con comportamiento intermedio. Funcio...,"Automatizaciones always-on, pruebas A/B de ofe..."
4,2,segmento_5,1117,565.33,1.04,"330,195.97","368,828,900.00",Dormidos de bajo retorno,Baja frecuencia y poca recencia. Conviene no s...,"Reactivacion de bajo costo, audiencias lookali..."


In [20]:
fig_segments = px.scatter(
    segments,
    x="frequency",
    y="monetary",
    color="business_label",
    size="recency",
    hover_name="customer_unique_id",
    title="Segmentos RFM: frecuencia vs monetario",
    labels={
        "frequency": "Frecuencia",
        "monetary": "Monetario COP",
        "business_label": "Segmento de negocio",
        "recency": "Recencia",
    },
)
fig_segments.update_layout(template="plotly_white")
fig_segments.show()

In [21]:
from IPython.display import Markdown, display

lines = [
    "### Lectura para VP de Marketing",
    f"El metodo del codo recomienda trabajar con **{selected_k} segmentos**. La lectura ejecutiva no se queda en el cluster tecnico: traduce cada grupo a una accion comercial.",
]

for _, row in business_explanation.sort_values("avg_monetary", ascending=False).iterrows():
    lines.append(
        f"**{row['business_label']}** ({int(row['customers']):,} clientes): "
        f"recencia promedio {row['avg_recency']:.0f} dias, "
        f"frecuencia promedio {row['avg_frequency']:.2f}, "
        f"monetario promedio ${row['avg_monetary']:,.0f} COP. "
        f"{row['vp_marketing_read']} Accion sugerida: {row['recommended_action']}"
    )

display(Markdown("\n\n".join(lines)))

### Lectura para VP de Marketing

El metodo del codo recomienda trabajar con **5 segmentos**. La lectura ejecutiva no se queda en el cluster tecnico: traduce cada grupo a una accion comercial.

**VIP recientes** (117 clientes): recencia promedio 130 dias, frecuencia promedio 3.84, monetario promedio $2,260,838 COP. Clientes de alto valor, compra reciente y mayor recurrencia relativa. Son la base de retencion premium. Accion sugerida: Proteger margen y lealtad: beneficios selectivos, acceso anticipado, cross-sell premium y programas de fidelizacion.

**Alto valor en riesgo** (365 clientes): recencia promedio 449 dias, frecuencia promedio 1.25, monetario promedio $1,383,332 COP. Compraron bien, pero llevan mas tiempo sin volver. Hay valor historico que puede perderse. Accion sugerida: Campanas win-back con incentivo controlado, mensajes personalizados y recordatorios por categoria comprada.

**Activos con potencial** (421 clientes): recencia promedio 216 dias, frecuencia promedio 2.24, monetario promedio $757,202 COP. Clientes recientes con buen gasto relativo, listos para aumentar frecuencia o ticket. Accion sugerida: Impulsar segunda compra, bundles, recomendaciones y comunicaciones por afinidad de categoria.

**Base estable** (1,123 clientes): recencia promedio 273 dias, frecuencia promedio 1.00, monetario promedio $335,563 COP. Clientes con comportamiento intermedio. Funcionan como base para nutricion y experimentos comerciales. Accion sugerida: Automatizaciones always-on, pruebas A/B de oferta y educacion de categorias complementarias.

**Dormidos de bajo retorno** (1,117 clientes): recencia promedio 565 dias, frecuencia promedio 1.04, monetario promedio $330,196 COP. Baja frecuencia y poca recencia. Conviene no sobreinvertir hasta validar respuesta. Accion sugerida: Reactivacion de bajo costo, audiencias lookalike separadas y supresion gradual si no responden.

## 13. Exportables

Guardamos los principales artefactos en `outputs/` para que queden disponibles fuera del notebook.

In [22]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

order_level.to_csv(OUTPUT_DIR / "order_level_dataset.csv", index=False)
monthly_sales.to_csv(OUTPUT_DIR / "ventas_por_mes.csv", index=False)
segments.to_csv(OUTPUT_DIR / "rfm_kmeans_segments.csv", index=False)
fig_sales.write_html(OUTPUT_DIR / "ventas_por_mes.html", include_plotlyjs="cdn")

print(f"Archivos exportados en: {OUTPUT_DIR}")

Archivos exportados en: c:\Users\jptru\platzi\spikes\spike-vscode-ia-datos\outputs
